[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/run_all.ipynb)

# ▶ Run All — both pipelines, condensed

Runs the whole project end-to-end with minimal commentary. For the **beginner-friendly,
fully-explained** versions, open `predictive_maintenance.ipynb` and
`anomaly_detection.ipynb`.

**Colab:** `Runtime → Run all`.

In [ ]:
# === Environment bootstrap — works on your laptop AND on Google Colab ========
# (Colab is a free website that runs Python notebooks in your browser, with a
#  free GPU. "GPU" = Graphics Processing Unit, a chip that makes ML training fast.)
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab the machine starts empty, so we download ("clone") the project
    # and install the libraries it needs.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Add the project folder to Python's search path so `from src import ...` works.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")
ROOT = _find_repo_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("Setup done. Running on", "Colab" if IN_COLAB else "your local machine.")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from src import data, features, models, utils
np.random.seed(0); results = {}
print("ready")

## 1 · Predictive maintenance — classification (AI4I)

In [ ]:
ai4i = data.load_ai4i(); X, y = features.prepare_ai4i(ai4i)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                             random_state=42, n_jobs=-1).fit(Xtr, ytr)
results["AI4I ROC-AUC"] = round(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]), 3)
print(classification_report(yte, clf.predict(Xte), digits=3))

## 2 · Predictive maintenance — RUL regression (C-MAPSS LSTM)

In [ ]:
cm = data.load_cmapss("FD001")
train_fe = features.add_rolling_features(features.add_rul(cm["train"], clip=125),
                                         features.feature_columns(cm["train"]))
cols = features.feature_columns(cm["train"])
Xs, ys = features.make_sequences(train_fe, cols, seq_len=30)
sc = utils.Standardizer().fit(Xs.reshape(-1, Xs.shape[-1]))
scale = lambda a: sc.transform(a.reshape(-1, a.shape[-1])).reshape(a.shape)
m = models.LSTMRegressor(n_features=Xs.shape[-1]); models.train_lstm(m, scale(Xs), ys, epochs=15)
yp = models.predict_lstm(m, scale(features.last_sequence_per_unit(cm["test"], cols, 30)))
yt = cm["rul"]["rul"].to_numpy().clip(max=125)
results["RUL RMSE (cycles)"] = round(utils.rmse(yt, yp), 1)
utils.plot_rul_scatter(yt, yp); plt.show()

## 3 · Anomaly detection (Isolation Forest + Autoencoder)

In [ ]:
df = train_fe; healthy = df[df.rul >= 100]; degraded = df[df.rul <= 20]
s = utils.Standardizer().fit(healthy[cols].to_numpy("float32"))
Xh, Xd = s.transform(healthy[cols].to_numpy("float32")), s.transform(degraded[cols].to_numpy("float32"))
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42).fit(Xh)
ae = models.AutoEncoder(n_features=Xh.shape[1]); models.train_autoencoder(ae, Xh, epochs=25, verbose=False)
lab = np.r_[np.zeros(len(Xh)), np.ones(len(Xd))]
results["Anomaly IsoForest AUC"] = round(roc_auc_score(lab, np.r_[-iso.decision_function(Xh), -iso.decision_function(Xd)]), 3)
results["Anomaly AE AUC"] = round(roc_auc_score(lab, np.r_[models.reconstruction_error(ae, Xh), models.reconstruction_error(ae, Xd)]), 3)
print("=== RESULTS ===")
for k, v in results.items(): print(f"  {k:26s}: {v}")